# DTCR-U-Net on Kaggle: MosMedData + LIDC-IDRI hybrid (paper Table 1 / Table 2)

End-to-end run of the paper's main experiment: pull the latest code, preprocess the two datasets, train for 80 epochs, evaluate on the held-out test patients, and bundle the logs.

**Before running**
1. *Settings -> Accelerator*: **GPU T4 x2** (or P100). *Settings -> Internet*: **On** (needed to clone the repository).
2. *Add Input*: attach **MosMedData** (the 50 annotated studies: `study_XXXX.nii.gz` + `study_XXXX_mask.nii.gz`) and **LIDC-IDRI** (raw DICOM + `LIDC-XML-only`, e.g. `washingtongold/lidcidri30`).
3. Do **not** attach COVID-19-CT-Seg for this run: the paper's 80.10 % Dice is reported on MosMedData + LIDC-IDRI only. It is ignored unless you set `INCLUDE_COVID19 = True` below.
4. Run all cells. When it finishes, download `/kaggle/working/dtcr_run_logs.zip` and share it.

## 1. Experiment settings

In [ ]:
# ---- Repository ----
REPO_URL = 'https://github.com/shadyOg/fyp-dtcr-unet.git'
BRANCH = 'main'

# ---- Data (paper Section 4.1) ----
INCLUDE_COVID19 = False      # keep False to reproduce the MosMedData + LIDC-IDRI hybrid
LIDC_PATIENTS = 100          # 50 MosMed + 100 LIDC = 150 patients -> 30 test patients (paper Table 2: n = 30)
IMAGE_SIZE = 256             # 512 x 512 slices downsampled to 256 x 256
CONTEXT_SLICES = 2           # each lesion slice plus its two neighbours above and below
FORCE_PREPROCESS = False     # True: rebuild the preprocessed data even if it already exists

# Set these only if auto-detection in section 3 picks the wrong folder
MOSMED_PATH = None
LIDC_PATH = None
COVID_PATH = None

# ---- Training (paper Section 4.1 / 4.6) ----
EPOCHS = 80
EFFECTIVE_BATCH = 32         # paper batch size; split into batch x accumulation steps to fit GPU memory
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABELED_RATIO = 0.4          # share of training patients that keep labels (the paper does not state it for this dataset)
LAMBDA1 = 0.3                # gradient-consistency weight (paper Fig. 7 optimum)
LAMBDA2 = 1.0                # interaction weight (Section 4.6)
BETA = 0.5                   # gradient term of the level-set loss (Section 4.6)
CONSISTENCY_WEIGHT = 1.0     # maximum of the ramp lambda_d(t) (Eq. 10)
SEED = 42

WORK = '/kaggle/working'
REPO_DIR = f'{WORK}/fyp-dtcr-unet'
PROCESSED = f'{WORK}/data/processed'
CKPT_DIR = f'{WORK}/checkpoints'
OUT_DIR = f'{WORK}/outputs'

## 2. GPU check, pull the latest code, install missing packages

In [ ]:
import os, subprocess, sys, importlib
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {p.name} | {p.total_memory / 1024**3:.1f} GB')
    print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
else:
    raise SystemExit('No GPU detected. Enable it under Settings -> Accelerator, then restart the session.')


def sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr).strip())
    if r.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')


# Clone on the first run, otherwise update to the newest commit of the branch
if os.path.isdir(f'{REPO_DIR}/.git'):
    sh(f'git -C {REPO_DIR} fetch --depth 1 origin {BRANCH}')
    sh(f'git -C {REPO_DIR} reset --hard FETCH_HEAD')
else:
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}')
sh(f'git -C {REPO_DIR} log -1 --format="commit %h | %ad | %s" --date=iso')

# Install only what the Kaggle image does not already have
needed = {'nibabel': 'nibabel', 'pydicom': 'pydicom', 'SimpleITK': 'SimpleITK', 'skimage': 'scikit-image',
          'sklearn': 'scikit-learn', 'cv2': 'opencv-python-headless', 'tqdm': 'tqdm', 'pandas': 'pandas'}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    sh(f'{sys.executable} -m pip install -q ' + ' '.join(missing))

CODE_DIR = f'{REPO_DIR}/dtcr-unet'
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

assert os.path.exists('data/level_set.py'), (
    'This checkout does not contain the latest fixes (data/level_set.py is missing). '
    'Push your local commits to GitHub, then re-run this cell.')
print('\nWorking directory:', os.getcwd())

## 3. Locate the input datasets
Only folder names are inspected (a few levels deep), so this stays fast even with the large LIDC-IDRI DICOM tree.

In [ ]:
import os, re
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')


def walk_dirs(root, max_depth=5):
    # Yield (dir_path, subdir_names, file_names) down to max_depth, without descending into DICOM patient folders
    root = Path(root)
    base = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).parts) - base
        yield Path(dirpath), dirnames, filenames
        if depth >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if not re.fullmatch(r'LIDC-IDRI-\d{4}', d)]


def top_level(path):
    # /kaggle/input/<dataset>/... -> /kaggle/input/<dataset>
    rel = Path(path).relative_to(INPUT_ROOT)
    return str(INPUT_ROOT / rel.parts[0])


found = {'mosmed': None, 'lidc': None, 'covid': None}
print('Attached inputs:', sorted(p.name for p in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else 'none')
for d, subdirs, files in walk_dirs(INPUT_ROOT):
    if found['mosmed'] is None and any(re.match(r'study_\d+_mask\.nii', f) for f in files):
        found['mosmed'] = top_level(d)
    if found['lidc'] is None and ('LIDC-XML-only' in subdirs or any(re.fullmatch(r'LIDC-IDRI-\d{4}-\d{4}', s) for s in subdirs)):
        found['lidc'] = str(d)
    if found['covid'] is None and 'infection_mask' in subdirs and 'ct_scans' in subdirs:
        found['covid'] = str(d)

MOSMED_PATH = MOSMED_PATH or found['mosmed']
LIDC_PATH = LIDC_PATH or found['lidc']
COVID_PATH = (COVID_PATH or found['covid']) if INCLUDE_COVID19 else None

print(f'MosMedData      : {MOSMED_PATH}')
print(f'LIDC-IDRI       : {LIDC_PATH}')
print(f'COVID-19-CT-Seg : {COVID_PATH}  (INCLUDE_COVID19 = {INCLUDE_COVID19})')
if found['covid'] and not INCLUDE_COVID19:
    print('  note: a COVID-19-CT-Seg input is attached but is NOT used in this run.')

assert MOSMED_PATH, 'MosMedData not found. Attach it via "Add Input" or set MOSMED_PATH in section 1.'
if not LIDC_PATH:
    print('\nWARNING: LIDC-IDRI not found. The run will use MosMedData only, which is not the paper\'s hybrid dataset.')

## 4. Preprocessing (paper Section 4.1)
Each volume is brought to one common orientation, windowed to [0, 1], contrast-enhanced (CLAHE) and resized to 256 x 256. Lesion slices and their two neighbours on each side are kept, and patients are split 6 : 2 : 2 into train / val / test. Which training patients are treated as unlabeled is decided later, at training time.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

wanted = {'mosmed': bool(MOSMED_PATH), 'lidc': bool(LIDC_PATH), 'covid19': bool(COVID_PATH),
          'size': IMAGE_SIZE, 'context': CONTEXT_SLICES, 'lidc_patients': LIDC_PATIENTS, 'seed': SEED}
marker = Path(PROCESSED) / 'run_settings.json'
up_to_date = marker.exists() and json.loads(marker.read_text()) == wanted

if up_to_date and not FORCE_PREPROCESS:
    print('Preprocessed data already matches these settings: skipping. (Set FORCE_PREPROCESS = True to rebuild.)')
else:
    shutil.rmtree(PROCESSED, ignore_errors=True)   # never mix slices from an older preprocessing run
    cmd = [sys.executable, '-u', 'data/preprocessing.py', '--out', PROCESSED, '--size', str(IMAGE_SIZE),
           '--context', str(CONTEXT_SLICES), '--max-patients', str(LIDC_PATIENTS), '--seed', str(SEED)]
    if MOSMED_PATH: cmd += ['--mosmed', MOSMED_PATH]
    if LIDC_PATH:   cmd += ['--lidc', LIDC_PATH]
    if COVID_PATH:  cmd += ['--covid19', COVID_PATH]
    print(' '.join(cmd), '\n')
    os.makedirs(OUT_DIR, exist_ok=True)
    with open(f'{OUT_DIR}/preprocess.log', 'w') as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                env={**os.environ, 'TQDM_MININTERVAL': '30'})
        for line in proc.stdout:
            log.write(line)
            print(line, end='')
        if proc.wait() != 0:
            raise RuntimeError('Preprocessing failed, see the output above.')
    marker.write_text(json.dumps(wanted))

print('\nSlices per split and source dataset:')
for split in ('train', 'val', 'test'):
    stems = [p.stem for p in Path(PROCESSED, split, 'images').glob('*.npy')]
    by_ds = {}
    for s in stems:
        ds = s.split('_', 1)[0]
        e = by_ds.setdefault(ds, [set(), 0])
        e[0].add(s.rsplit('_s', 1)[0]); e[1] += 1
    print(f'  {split:<5}: ' + ' | '.join(f'{ds}: {len(p)} patients, {n} slices' for ds, (p, n) in sorted(by_ds.items())))
    assert stems, f'No slices were produced for the {split} split.'

## 5. Look at the preprocessed data
A quick visual check before spending GPU hours: lesion outlines (red) should sit on the lesions, and scans from both datasets should be the same way up (spine at the bottom).

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

rng = random.Random(0)
train_dir = Path(PROCESSED, 'train')
by_ds = {}
for p in sorted(train_dir.glob('masks/*.npy')):
    by_ds.setdefault(p.stem.split('_', 1)[0], []).append(p)

for ds, paths in sorted(by_ds.items()):
    areas = [int(np.load(p).sum()) for p in paths]
    lesion = [(p, a) for p, a in zip(paths, areas) if a > 0]
    med = np.median([a for _, a in lesion]) if lesion else 0
    print(f'{ds}: {len(paths)} train slices, {len(lesion)} with lesion, median lesion area {med:.0f} px '
          f'({100 * med / IMAGE_SIZE**2:.2f} % of the slice)')
    picks = rng.sample(lesion, min(6, len(lesion)))
    fig, axes = plt.subplots(1, len(picks), figsize=(3.2 * len(picks), 3.4), squeeze=False)
    for ax, (p, a) in zip(axes[0], picks):
        img = np.load(train_dir / 'images' / p.name)
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.contour(np.load(p), levels=[0.5], colors='red', linewidths=0.8)
        ax.set_title(f'{ds} | {a} px', fontsize=9)
        ax.axis('off')
    plt.tight_layout(); plt.show()

## 6. Pick the largest batch that fits on this GPU
The paper trains with batch size 32. If 32 does not fit in memory, a smaller batch is used with gradient accumulation so that each optimizer step still sees 32 slices.

In [ ]:
import gc, torch
from models.dtcr_unet import DTCR_UNet
from losses.dtc_loss import DTCRTotalLoss


def peak_memory_gb(batch_size):
    model = DTCR_UNet().cuda().train()
    crit = DTCRTotalLoss().cuda()
    x = torch.rand(batch_size, 1, IMAGE_SIZE, IMAGE_SIZE, device='cuda')
    y = (torch.rand_like(x) > 0.9).float()
    lab = torch.arange(batch_size, device='cuda') < batch_size // 2
    torch.cuda.reset_peak_memory_stats()
    with torch.amp.autocast('cuda'):
        f1, f2, f2t = model(x)
        loss, _ = crit(f1, f2, f2t, y, 1 - 2 * y, lab, epoch=1, max_epochs=2)
    loss.backward()
    return torch.cuda.max_memory_allocated() / 1024**3


def fits(batch_size):
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    try:
        peak = peak_memory_gb(batch_size)
    except torch.cuda.OutOfMemoryError:
        peak = None
    gc.collect(); torch.cuda.empty_cache()
    if peak is None:
        print(f'  batch {batch_size}: out of memory')
        return False
    print(f'  batch {batch_size}: peak {peak:.1f} GB of {total:.1f} GB')
    return peak < 0.85 * total          # leave headroom for validation and fragmentation


BATCH_SIZE = next((b for b in (32, 16, 8, 4) if b <= EFFECTIVE_BATCH and fits(b)), 4)
GRAD_ACCUM = max(1, EFFECTIVE_BATCH // BATCH_SIZE)
gc.collect(); torch.cuda.empty_cache()
print(f'\nUsing batch size {BATCH_SIZE} x {GRAD_ACCUM} accumulation step(s) = effective batch {BATCH_SIZE * GRAD_ACCUM}')

## 7. Train
AdamW, learning rate 1e-3 with cosine annealing, 80 epochs, checkpoint with the lowest validation loss kept as `best_model.pth` (paper Section 4.1). The checkpoint with the highest validation Dice is also kept as `best_dice_model.pth`.

Each epoch prints one line. **PATIENT DICE** is the paper's metric (Dice per patient volume, averaged over patients); the bracket shows it per source dataset. The full log is written to `outputs/train.log`.

In [ ]:
import os, subprocess, sys

os.makedirs(OUT_DIR, exist_ok=True)
cmd = [sys.executable, '-u', 'train.py',
       '--data', PROCESSED, '--num-stages', '4',
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--grad-accum', str(GRAD_ACCUM),
       '--lr', str(LR), '--weight-decay', str(WEIGHT_DECAY),
       '--labeled-ratio', str(LABELED_RATIO),
       '--beta', str(BETA), '--lambda1', str(LAMBDA1), '--lambda2', str(LAMBDA2),
       '--consistency-weight', str(CONSISTENCY_WEIGHT),
       '--num-workers', str(min(4, os.cpu_count() or 2)), '--seed', str(SEED),
       '--checkpoints', CKPT_DIR, '--outputs', OUT_DIR]
print(' '.join(cmd), '\n')

with open(f'{OUT_DIR}/train.log', 'w') as log:
    log.write(' '.join(cmd) + '\n')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'TQDM_DISABLE': '1'})   # one clean line per epoch
    for line in proc.stdout:
        log.write(line); log.flush()
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError('Training failed, see the output above.')

## 8. Training curves

In [ ]:
import json
import matplotlib.pyplot as plt

hist = json.load(open(f'{OUT_DIR}/training_history.json'))
ep = [h['epoch'] for h in hist]
tl = lambda k: [h['train_losses'][k] for h in hist]
vm = lambda k: [h['val_metrics'][k] for h in hist]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].plot(ep, tl('total'), label='train total')
axes[0].plot(ep, vm('val_loss'), label='val total')
axes[0].plot(ep, tl('seg'), '--', label='train seg (Dice loss)')
axes[0].plot(ep, vm('val_seg'), '--', label='val seg (Dice loss)')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(ep, vm('dice'), lw=2.5, label='patient Dice (all)')
for k in sorted(hist[0]['val_metrics']):
    if k.startswith('dice_') and k != 'dice_std':
        axes[1].plot(ep, vm(k), label=f'patient Dice: {k[5:]}')
axes[1].plot(ep, vm('lesion_dice'), ':', label='per-slice Dice (lesion slices)')
axes[1].axhline(80.10, color='gray', ls='--', lw=1); axes[1].text(ep[0], 80.8, 'paper: 80.10', color='gray', fontsize=9)
axes[1].set_title('Validation Dice (%)'); axes[1].set_xlabel('epoch'); axes[1].set_ylim(0, 100); axes[1].legend()

axes[2].plot(ep, tl('lsf'), label='level-set loss')
axes[2].plot(ep, tl('dtc'), label='consistency loss L_DTC')
axes[2].plot(ep, tl('lambda_d'), label='lambda_d(t)')
axes[2].plot(ep, [v / 100 for v in vm('task_agreement')], label='agreement of the two heads')
axes[2].set_title('Boundary task and consistency'); axes[2].set_xlabel('epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/training_curves.png', dpi=120); plt.show()

best_loss = min(hist, key=lambda h: h['val_metrics']['val_loss'])
best_dice = max(hist, key=lambda h: h['val_metrics']['dice'])
print(f"Lowest val loss : epoch {best_loss['epoch']}, patient Dice {best_loss['val_metrics']['dice']:.2f} %")
print(f"Highest val Dice: epoch {best_dice['epoch']}, patient Dice {best_dice['val_metrics']['dice']:.2f} %")

## 9. Test-set evaluation
Evaluated on the held-out test patients, which were never used for training or model selection. `best_model.pth` (lowest validation loss) is the paper's selection rule; `best_dice_model.pth` is reported alongside for comparison.

In [ ]:
import os, subprocess, sys

for name in ('best_model', 'best_dice_model'):
    ckpt = f'{CKPT_DIR}/{name}.pth'
    if not os.path.exists(ckpt):
        print(f'{ckpt} not found, skipping.'); continue
    print('\n' + '#' * 70 + f'\n# {name}.pth\n' + '#' * 70)
    cmd = [sys.executable, '-u', 'evaluate.py', '--checkpoint', ckpt, '--data', PROCESSED,
           '--batch-size', str(BATCH_SIZE), '--threshold', '0.5', '--save-vis', '16',
           '--out', f'{WORK}/evaluation_{name}']
    r = subprocess.run(cmd, text=True, capture_output=True, env={**os.environ, 'TQDM_DISABLE': '1'})
    print(r.stdout); print(r.stderr[-3000:] if r.returncode else '')
    open(f'{OUT_DIR}/evaluation_{name}.log', 'w').write(r.stdout + r.stderr)

## 10. Example predictions on test slices

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

images = sorted(glob.glob(f'{WORK}/evaluation_best_model/comparisons/*.png'))[:8]
if not images:
    print('No comparison images found.')
for path in images:
    plt.figure(figsize=(18, 4.5)); plt.imshow(Image.open(path)); plt.axis('off'); plt.show()

## 11. Bundle the logs
Creates `/kaggle/working/dtcr_run_logs.zip` (logs, per-epoch history, curves and test metrics; no model weights). Download it from the *Output* panel and share it.

In [ ]:
import glob, json, os, zipfile

bundle = f'{WORK}/dtcr_run_logs.zip'
files = (glob.glob(f'{OUT_DIR}/*.log') + glob.glob(f'{OUT_DIR}/*.json') + glob.glob(f'{OUT_DIR}/*.png')
         + glob.glob(f'{PROCESSED}/*.json') + glob.glob(f'{WORK}/evaluation_*/test_metrics.json'))
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, os.path.relpath(f, WORK))
print(f'Wrote {bundle} ({os.path.getsize(bundle) / 1024:.0f} KB, {len(files)} files)\n')

for name in ('best_model', 'best_dice_model'):
    path = f'{WORK}/evaluation_{name}/test_metrics.json'
    if os.path.exists(path):
        r = json.load(open(path))
        p = r['patient_level_3d']
        per_ds = ' | '.join(f"{ds}: {v['dice_mean']:.2f} ({v['patients']} pts)" for ds, v in r['patient_level_by_dataset'].items())
        print(f"{name:<16} TEST patient Dice {p['dice']['mean']:.2f} +- {p['dice']['std']:.2f} % over {r['total_patients']} patients "
              f"| SE {p['sensitivity']['mean']:.2f} SP {p['specificity']['mean']:.2f} Acc {p['accuracy']['mean']:.2f} | {per_ds}")
print('\nPaper (Table 1 / 2): Dice 80.10 +- 2.41 %, SE 92.37 %, SP 99.87 %, Accuracy 95.68 % (n = 30 patients)')